# Chapter 3: Convolutional Neural Networks (CNNs from Scratch)

Welcome to **Chapter 3**!

In Chapters 1 and 2, we worked with **Multi-Layer Perceptrons (MLPs)** where every single pixel was connected to every neuron. We saw that an MLP reached ~87.6% accuracy on Fashion-MNIST, but frequently confused **Shirts**, **T-shirts**, and **Coats**.

Why? Because an MLP has a fatal blind spot: **it destroys the 2D spatial arrangement of the image!**
If a collar, sleeve, or button shifts by just 2 pixels, an MLP treats it as an entirely new feature and has to learn it from scratch.

In this chapter, we build a **Convolutional Neural Network (CNN)** from first principles in pure NumPy:
1. **2D Convolutions & Parameter Sharing**: Sliding a tiny $3 \times 3$ filter across the image to detect edges, seams, and textures regardless of where they appear.
2. **Spatial Max Pooling**: Shrinking spatial dimensions while preserving salient features.
3. **Vectorized `im2col` & `col2im`**: Turning slow 4D sliding loops into blazing-fast matrix multiplications.
4. **End-to-End Training**: Training a 2-stage CNN on Fashion-MNIST with Adam.

---

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dataset import load_fashion_mnist, FASHION_MNIST_CLASSES, get_batches
from src.cnn import Conv2D, MaxPool2D, Flatten, ConvNet
from src.optimizers import Adam

print("Chapter 3 CNN modules loaded from:", repo_root)

## 1. The Power of 2D Convolutions: Hand-Crafted Edge Detectors

Before training a network to learn filters automatically, let us see how a tiny $3 \times 3$ matrix of numbers can detect physical features like horizontal lines, vertical lines, or all-direction outlines.

Let us apply **Sobel filters** to a sample clothing item:

In [ ]:
# Load a single sample garment
X_train, y_train, _, _, _, _ = load_fashion_mnist(flatten=False, normalize=True)
sample_idx = int(np.where(y_train == 6)[0][0])  # A Shirt
shirt_img = X_train[sample_idx:sample_idx + 1, np.newaxis, :, :]  # (1, 1, 28, 28)

# 1. Horizontal edge filter (Sobel H)
sobel_h = np.array([[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]], dtype=np.float32).reshape(1, 1, 3, 3)

# 2. Vertical edge filter (Sobel V)
sobel_v = np.array([[-1,  0,  1],
                    [-2,  0,  2],
                    [-1,  0,  1]], dtype=np.float32).reshape(1, 1, 3, 3)

# 3. Outline filter (Laplacian)
laplacian = np.array([[ 0,  1,  0],
                      [ 1, -4,  1],
                      [ 0,  1,  0]], dtype=np.float32).reshape(1, 1, 3, 3)

filters = [("Horizontal Edges", sobel_h), ("Vertical Edges", sobel_v), ("Outlines", laplacian)]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(shirt_img[0, 0], cmap="gray_r")
axes[0].set_title("Original Shirt (28x28)", fontweight="bold")
axes[0].axis("off")

for idx, (title, k) in enumerate(filters, start=1):
    c = Conv2D(1, 1, kernel_size=3, padding=1)
    c.W = k.copy()
    c.b = np.zeros((1, 1), dtype=np.float32)
    fmap = c.forward(shirt_img)[0, 0]
    axes[idx].imshow(fmap, cmap="viridis")
    axes[idx].set_title(title, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Feature Maps Extracted by 3x3 Convolutional Kernels", fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

## 2. Spatial Max Pooling: Downsampling with Translation Invariance

Max Pooling takes a $2 \times 2$ block of activations and keeps only the maximum value.
This halves the height and width ($28 \times 28 \to 14 \times 14$), discarding small spatial jitter while keeping the strongest signal.

In [ ]:
pool = MaxPool2D(pool_size=2, stride=2)
pooled_shirt = pool.forward(shirt_img)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(shirt_img[0, 0], cmap="gray_r")
ax1.set_title(f"Before Pooling: {shirt_img.shape[2:]} = 784 values", fontweight="bold")
ax1.axis("off")

ax2.imshow(pooled_shirt[0, 0], cmap="gray_r")
ax2.set_title(f"After 2x2 MaxPool: {pooled_shirt.shape[2:]} = 196 values", fontweight="bold")
ax2.axis("off")

plt.tight_layout()
plt.show()

## 3. End-to-End ConvNet Training on Fashion-MNIST

We assemble our full 2-stage CNN:
$$\text{Input } (N, 1, 28, 28) \to \text{Conv2D}(8) \to \text{ReLU} \to \text{MaxPool} (14\times 14) \to \text{Conv2D}(16) \to \text{ReLU} \to \text{MaxPool} (7\times 7) \to \text{FC}(64) \to \text{Softmax}(10)$$

In [ ]:
# Load datasets
X_train, y_train, X_val, y_val, X_test, y_test = load_fashion_mnist(flatten=False, normalize=True)
X_train = X_train[:, np.newaxis, :, :]
X_val = X_val[:, np.newaxis, :, :]
X_test = X_test[:, np.newaxis, :, :]

model = ConvNet(in_channels=1, conv1_channels=8, conv2_channels=16, hidden_dim=64, num_classes=10, seed=42)
optimizer = Adam(lr=0.002)

epochs = 3
batch_size = 128
train_sub = 15000

print(f"Training ConvNet on {train_sub:,} Fashion-MNIST images...")
for ep in range(1, epochs + 1):
    perm = np.random.permutation(train_sub)
    for i in range(0, train_sub, batch_size):
        b_idx = perm[i:i + batch_size]
        X_b, y_b = X_train[b_idx], y_train[b_idx]
        model.forward(X_b)
        grads = model.backward(y_b)
        for p in model.params.keys():
            model.set_param(p, optimizer.step(p, model.params[p], grads[p]))
    
    val_loss, val_acc = model.evaluate(X_val[:3000], y_val[:3000])
    print(f"  Epoch {ep}/{epochs} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc*100:.2f}%")

## 4. Visualizing Internal Feature Maps

Let us pass a test garment through our trained ConvNet and inspect what the 8 filters in Conv1 and the 16 filters in Conv2 have learned to detect:

In [ ]:
test_idx = int(np.where(y_test == 9)[0][0])  # Ankle boot
boot_img = X_test[test_idx:test_idx + 1]
model.forward(boot_img)

a1 = model.cache["a1"][0]
a2 = model.cache["a2"][0]

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i in range(8):
    axes[0, i].imshow(a1[i], cmap="magma")
    axes[0, i].set_title(f"Conv1 #{i+1}", fontsize=9)
    axes[0, i].axis("off")
    
    axes[1, i].imshow(a2[i], cmap="viridis")
    axes[1, i].set_title(f"Conv2 #{i+1}", fontsize=9)
    axes[1, i].axis("off")

plt.suptitle("Feature Maps: Layer 1 (Sharp Edges) -> Layer 2 (Abstract Silhouettes)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5. Unseen Test Accuracy Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Final ConvNet Test Accuracy on 10,000 Unseen Garments: {test_acc * 100:.2f}%")